In [60]:
# all imports
import pyBigWig
import pandas as pd
import numpy as np

In [61]:
# func
def extract_mean_signal(bw, row):
    values = bw.values(row["chrom"], int(row["start"]), int(row["end"]), numpy=True)
    return np.nanmean(values)

In [62]:
# import all loops
all_loops = pd.read_csv("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/ml_2_multiclassmodel.csv")
all_loops

,Unnamed: 0,chr1,start1,end1,chr2,start2,end2,status,ctrl_signal,rbp1_signal,log2FC,log2FC_clipped,condition,distance,loop_id,comp_switch_A1,comp_switch_A2,loop_class,source
0,0,chr1,1952500,1957500,chr1,2042500,2047500,shared,NaN,NaN,0.000000,0.000000,shared,90000,0,NaN,NaN,uncategorized,ctrl
1,1,chr1,2202500,2207500,chr1,2382500,2387500,shared,NaN,NaN,0.000000,0.000000,shared,180000,1,NaN,NaN,uncategorized,ctrl
2,2,chr1,2412500,2417500,chr1,2552500,2557500,shared,NaN,NaN,0.000000,0.000000,shared,140000,2,NaN,NaN,uncategorized,ctrl
3,3,chr1,3490000,3495000,chr1,3615000,3620000,shared,NaN,NaN,0.000000,0.000000,shared,125000,3,NaN,NaN,uncategorized,ctrl
4,4,chr1,3565000,3570000,chr1,3615000,3620000,shared,NaN,NaN,0.000000,0.000000,shared,50000,4,NaN,NaN,uncategorized,ctrl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32272,32272,chrY,10945000,10950000,chrY,11290000,11295000,gained,0.006078,0.006342,0.061177,0.061177,Mutant loop,345000,32272,stable,stable,intra-TAD,rbp1
32273,32273,chrY,10982500,10987500,chrY,11292500,11297500,gained,0.007661,0.005159,-0.570544,-0.570544,Mutant loop,310000,32273,stable,stable,intra-TAD,rbp1
32274,32274,chrY,11292500,11297500,chrY,11722500,11727500,gained,0.001017,0.001021,0.005836,0.005836,Mutant loop,430000,32274,stable,stable,uncategorized,rbp1
32275,32275,chrY,11530000,11535000,chrY,11760000,11765000,gained,0.004674,0.006621,0.502513,0.502513,Mutant loop,230000,32275,stable,stable,intra-TAD,rbp1


In [63]:
# split to anchor1 and anchor2 bed and concat
anchor1 = all_loops[["chr1", "start1", "end1"]].copy()
anchor2 = all_loops[["chr2", "start2", "end2"]].copy()
#rename for consistency
anchor1.columns = ["chrom", "start", "end"]
anchor2.columns = ["chrom", "start", "end"]
#add 4th col - uniq
anchor1["name"] = [f"anchor1_{i}" for i in range(len(anchor1))]
anchor2["name"] = [f"anchor2_{i}" for i in range(len(anchor2))]
#concat
anchors_bed = pd.concat([anchor1, anchor2], axis=0).reset_index(drop=True)
anchors_bed

,chrom,start,end,name
0,chr1,1952500,1957500,anchor1_0
1,chr1,2202500,2207500,anchor1_1
2,chr1,2412500,2417500,anchor1_2
3,chr1,3490000,3495000,anchor1_3
4,chr1,3565000,3570000,anchor1_4
...,...,...,...,...
64549,chrY,11290000,11295000,anchor2_32272
64550,chrY,11292500,11297500,anchor2_32273
64551,chrY,11722500,11727500,anchor2_32274
64552,chrY,11760000,11765000,anchor2_32275


In [64]:
#ctcf
# open bigwig
ctcf_bw = pyBigWig.open("/usr/users/papantonis1/aman/microc_data/nadine_macro/C_CTCF.bw_RPGC.bw")
ctcf_bw.isBigWig()

True

In [65]:
anchors_bed["CTCF_signal"] = anchors_bed.apply(lambda row: extract_mean_signal(ctcf_bw, row), axis=1)
anchors_bed

,chrom,start,end,name,CTCF_signal
0,chr1,1952500,1957500,anchor1_0,4.046024
1,chr1,2202500,2207500,anchor1_1,3.393440
2,chr1,2412500,2417500,anchor1_2,63.496483
3,chr1,3490000,3495000,anchor1_3,60.820889
4,chr1,3565000,3570000,anchor1_4,23.754074
...,...,...,...,...,...
64549,chrY,11290000,11295000,anchor2_32272,0.000000
64550,chrY,11292500,11297500,anchor2_32273,0.522068
64551,chrY,11722500,11727500,anchor2_32274,0.065258
64552,chrY,11760000,11765000,anchor2_32275,0.000000


In [66]:
# h3k27me3
h3k27me3_bw = pyBigWig.open("/usr/users/papantonis1/aman/microc_data/cut_n_tag/nadine_cut_tag/nadine_cut_tag/C_H3K27me3_results/Aligned_files/Bigwig_scaled/C_H3K27me3.bw_RPGC.bw")
h3k27me3_bw.isBigWig()

True

In [67]:
anchors_bed["H3K27me3_signal"] = anchors_bed.apply(lambda row: extract_mean_signal(h3k27me3_bw, row), axis=1)
anchors_bed

,chrom,start,end,name,CTCF_signal,H3K27me3_signal
0,chr1,1952500,1957500,anchor1_0,4.046024,1.318393
1,chr1,2202500,2207500,anchor1_1,3.393440,2.542615
2,chr1,2412500,2417500,anchor1_2,63.496483,1.789249
3,chr1,3490000,3495000,anchor1_3,60.820889,1.294850
4,chr1,3565000,3570000,anchor1_4,23.754074,1.318393
...,...,...,...,...,...,...
64549,chrY,11290000,11295000,anchor2_32272,0.000000,0.164799
64550,chrY,11292500,11297500,anchor2_32273,0.522068,0.376683
64551,chrY,11722500,11727500,anchor2_32274,0.065258,0.000000
64552,chrY,11760000,11765000,anchor2_32275,0.000000,0.188342


In [68]:
# h3k4me3
h3k4me3_bw = pyBigWig.open("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/DLD1_H3K4me3.bw")
h3k4me3_bw.isBigWig()

True

In [69]:
anchors_bed["H3K4me3_signal"] = anchors_bed.apply(lambda row: extract_mean_signal(h3k4me3_bw, row), axis=1)
anchors_bed

/tmp/ipykernel_248434/3037688977.py:4: RuntimeWarning: Mean of empty slice
  return np.nanmean(values)


,chrom,start,end,name,CTCF_signal,H3K27me3_signal,H3K4me3_signal
0,chr1,1952500,1957500,anchor1_0,4.046024,1.318393,0.188543
1,chr1,2202500,2207500,anchor1_1,3.393440,2.542615,0.188543
2,chr1,2412500,2417500,anchor1_2,63.496483,1.789249,0.520257
3,chr1,3490000,3495000,anchor1_3,60.820889,1.294850,0.611992
4,chr1,3565000,3570000,anchor1_4,23.754074,1.318393,NaN
...,...,...,...,...,...,...,...
64549,chrY,11290000,11295000,anchor2_32272,0.000000,0.164799,0.202684
64550,chrY,11292500,11297500,anchor2_32273,0.522068,0.376683,0.218400
64551,chrY,11722500,11727500,anchor2_32274,0.065258,0.000000,0.210467
64552,chrY,11760000,11765000,anchor2_32275,0.000000,0.188342,0.188543


In [70]:
# cohesin - rad21
rad21_bw = pyBigWig.open("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/cohesin_rad21.bw")
rad21_bw.isBigWig()

True

In [71]:
anchors_bed["rad21_signal"] = anchors_bed.apply(lambda row: extract_mean_signal(rad21_bw, row), axis=1)
anchors_bed

/tmp/ipykernel_248434/3037688977.py:4: RuntimeWarning: Mean of empty slice
  return np.nanmean(values)


,chrom,start,end,name,CTCF_signal,H3K27me3_signal,H3K4me3_signal,rad21_signal
0,chr1,1952500,1957500,anchor1_0,4.046024,1.318393,0.188543,0.025058
1,chr1,2202500,2207500,anchor1_1,3.393440,2.542615,0.188543,0.022979
2,chr1,2412500,2417500,anchor1_2,63.496483,1.789249,0.520257,0.070697
3,chr1,3490000,3495000,anchor1_3,60.820889,1.294850,0.611992,0.056158
4,chr1,3565000,3570000,anchor1_4,23.754074,1.318393,NaN,0.066047
...,...,...,...,...,...,...,...,...
64549,chrY,11290000,11295000,anchor2_32272,0.000000,0.164799,0.202684,0.108440
64550,chrY,11292500,11297500,anchor2_32273,0.522068,0.376683,0.218400,0.121345
64551,chrY,11722500,11727500,anchor2_32274,0.065258,0.000000,0.210467,0.104138
64552,chrY,11760000,11765000,anchor2_32275,0.000000,0.188342,0.188543,0.022683


In [72]:
# append back to all_loops
# Average anchor1 and anchor2 signals per loop (i.e., every 2 rows in anchors_bed)
avg_signals = anchors_bed[["CTCF_signal", "H3K27me3_signal", "H3K4me3_signal", "rad21_signal"]] \
                .groupby(anchors_bed.index // 2).mean()

# Now assign to all_loops (row counts will match: 32,277)
all_loops[["CTCF_signal", "H3K27me3_signal", "H3K4me3_signal", "rad21_signal"]] = avg_signals.values

In [73]:
all_loops

,Unnamed: 0,chr1,start1,end1,chr2,start2,end2,status,ctrl_signal,rbp1_signal,...,distance,loop_id,comp_switch_A1,comp_switch_A2,loop_class,source,CTCF_signal,H3K27me3_signal,H3K4me3_signal,rad21_signal
0,0,chr1,1952500,1957500,chr1,2042500,2047500,shared,NaN,NaN,...,90000,0,NaN,NaN,uncategorized,ctrl,3.719732,1.930504,0.188543,0.024018
1,1,chr1,2202500,2207500,chr1,2382500,2387500,shared,NaN,NaN,...,180000,1,NaN,NaN,uncategorized,ctrl,62.158684,1.542050,0.566125,0.063427
2,2,chr1,2412500,2417500,chr1,2552500,2557500,shared,NaN,NaN,...,140000,2,NaN,NaN,uncategorized,ctrl,13.704273,4.520206,0.188543,0.044699
3,3,chr1,3490000,3495000,chr1,3615000,3620000,shared,NaN,NaN,...,125000,3,NaN,NaN,uncategorized,ctrl,1.076765,5.414830,0.188543,0.024769
4,4,chr1,3565000,3570000,chr1,3615000,3620000,shared,NaN,NaN,...,50000,4,NaN,NaN,uncategorized,ctrl,16.477770,1.377250,0.188543,0.042592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32272,32272,chrY,10945000,10950000,chrY,11290000,11295000,gained,0.006078,0.006342,...,345000,32272,stable,stable,intra-TAD,rbp1,0.000000,0.000000,0.188543,0.026824
32273,32273,chrY,10982500,10987500,chrY,11292500,11297500,gained,0.007661,0.005159,...,310000,32273,stable,stable,intra-TAD,rbp1,0.522068,0.188342,0.249019,0.130330
32274,32274,chrY,11292500,11297500,chrY,11722500,11727500,gained,0.001017,0.001021,...,430000,32274,stable,stable,uncategorized,rbp1,0.261034,0.082400,0.241160,0.123878
32275,32275,chrY,11530000,11535000,chrY,11760000,11765000,gained,0.004674,0.006621,...,230000,32275,stable,stable,intra-TAD,rbp1,0.293663,0.188342,0.214433,0.112741


In [74]:
all_loops = all_loops.drop(columns=["Unnamed: 0"])

In [75]:
#export
all_loops.to_csv("ml_3_multiclassmodel.csv")

In [76]:
# h3k27ac

In [77]:
# ATAC